In [ ]:
import os, json, glob
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)

RF = os.path.join(ROOT, "final-results", "rf_study")
RC = os.path.join(ROOT, "final-results", "rf_corrected")
FIGS = os.path.join(ROOT, "final-results", "figures")
os.makedirs(FIGS, exist_ok=True)

def save(fig, name):
    for ext in ["svg", "pdf"]:
        fig.savefig(os.path.join(FIGS, name + "." + ext))

orange, red, blue, green, purple, grey = "#FF7F0E", "#c00000", "#1F77B4", "#2ca02c", "#7e57c2", "#888888"
plt.rcParams.update({"font.size": 12, "axes.spines.top": False})

In [ ]:
def load(pat):
    out = {}
    for f in glob.glob(os.path.join(RF, pat)):
        d = json.load(open(f))
        out.setdefault(d["beta"], {})[d["seed"]] = d
    return out

allb = {}
for b, sd in list(load("sweep_b*_s42.json").items()) + list(load("final_b*_s4[34].json").items()):
    allb.setdefault(b, {}).update(sd)

betas = sorted([b for b in allb if b > 0])
b0 = allb[0.0]

def s42(b, *keys):
    d = allb[b][42] if 42 in allb[b] else list(allb[b].values())[0]
    for k in keys:
        d = d[k]
    return d

print("betas:", betas)

In [ ]:
fig, (axA, axB) = plt.subplots(1, 2, figsize=(12, 4.6))

dead = [s42(b, "gate_stats", "dead_frac") * 100 for b in betas]
const = [s42(b, "gate_stats", "const_frac") * 100 for b in betas]
soft = [s42(b, "soft_test") * 100 for b in betas]
circ = [s42(b, "circuit_test") * 100 for b in betas]

axA.plot(betas, dead, "o-", color=red, lw=2.2, label="dead (all-UNK) gates")
axA.plot(betas, const, "s--", color="#e08", lw=1.4, ms=4, label="constant gates")
axA.axhline(b0[42]["gate_stats"]["dead_frac"] * 100, color=red, ls=":", lw=1, alpha=.6)
for b in betas:
    for s in (43, 44):
        if s in allb[b]:
            axA.plot(b, allb[b][s]["gate_stats"]["dead_frac"] * 100, "o", color=red, mfc="white", ms=5, alpha=.8)
axA.set_xscale("log")
axA.set_xlabel(r"R$_F$ weight  $\beta$  (0 = baseline, dotted)")
axA.set_ylabel("% of neurons → dead / constant gate", color=red)
axA.tick_params(axis="y", labelcolor=red)
axA.annotate("even β=6e-4 → " + str(round(dead[0])) + "%\n(baseline " + str(round(b0[42]['gate_stats']['dead_frac']*100, 2)) + "%)",
             xy=(betas[0], dead[0]), xytext=(betas[1] * 1.3, 38), color=red, fontsize=9.5,
             arrowprops=dict(arrowstyle="->", color=red))

axA2 = axA.twinx(); axA2.spines["top"].set_visible(False)
axA2.plot(betas, soft, "^-", color=blue, lw=1.8, label="soft test acc")
axA2.plot(betas, circ, "v-", color=green, lw=1.8, label="circuit test acc")
axA2.axhline(b0[42]["soft_test"] * 100, color=blue, ls=":", lw=1, alpha=.5)
axA2.set_ylabel("test accuracy (%)", color=blue); axA2.tick_params(axis="y", labelcolor=blue)
axA.set_title("R$_F$'s optimum IS the dead gate\n(accuracy rises — but by pruning, not interpretability)", fontsize=11)
l1, la = axA.get_legend_handles_labels(); l2, lb = axA2.get_legend_handles_labels()
axA.legend(l1 + l2, la + lb, fontsize=8, loc="lower right", frameon=False)

rawl1 = [s42(b, "hard_spectrum", "mean_l1_norm") for b in betas]
shape = [s42(b, "soft_spectrum", "rf_shape") for b in betas]
ttl2 = [s42(b, "soft_spectrum", "mean_tt_l2") for b in betas]
axB.plot(betas, rawl1, "o-", color=orange, lw=2.2, label="raw Fourier L1 (hardened) — what R$_F$ targets")
axB.plot(betas, ttl2, "D--", color=grey, lw=1.4, ms=4, label=r"coeff scale  $\langle\|tt\|_2\rangle$")
axB.plot(betas, shape, "s-", color=purple, lw=2.2, label="scale-invariant Fourier L1 (shape)")
axB.axhline(b0[42]["hard_spectrum"]["mean_l1_norm"], color=orange, ls=":", lw=1, alpha=.5)
axB.set_xscale("log"); axB.set_xlabel(r"R$_F$ weight  $\beta$")
axB.set_ylabel("Fourier L1 / scale")
axB.set_title("The L1 drop is coefficient-scale shrink + gate death,\nNOT spectral-shape sparsification", fontsize=11)
axB.legend(fontsize=8, loc="upper right", frameon=False)

fig.tight_layout()
save(fig, "rf_ablation")

In [ ]:
for b in [0.0, 6e-4, 6e-2]:
    if b not in allb:
        continue
    sd = allb[b]
    def ms(*k):
        v = []
        for s, d in sd.items():
            x = d
            for kk in k:
                x = x[kk]
            v.append(x)
        return np.mean(v), np.std(v), len(v)
    st, ct, dd = ms("soft_test"), ms("circuit_test"), ms("gate_stats", "dead_frac")
    print("beta", b, "n", st[2], "soft", round(st[0]*100, 2), "circ", round(ct[0]*100, 2),
          "dead", round(dd[0]*100, 2))

In [ ]:
d = json.load(open(os.path.join(RC, "single_gate.json")))

ms_rows = [5, 6, 7]
order = ["none", "L2", "ttL1", "raw_rf", "rf_floor"]
col = {"none": "#888888", "L2": "#348ABD", "ttL1": "#c00000",
       "raw_rf": "#FFB36E", "rf_floor": "#E24A33"}
lab = {"none": "no reg", "L2": "L2 (control)", "ttL1": "plain ttL1 (control)",
       "raw_rf": "raw Fourier L1", "rf_floor": "corrected R$_F^\\star$ (floor)"}

fig, (axA, axB) = plt.subplots(1, 2, figsize=(12, 4.6))

for k in order:
    g = [d["m" + str(m)][k]["gen"] for m in ms_rows]
    e = [d["m" + str(m)][k]["gen_std"] for m in ms_rows]
    axA.errorbar(ms_rows, g, yerr=e, marker="o", color=col[k],
                 lw=2.2 if "rf" in k else 1.6, label=lab[k], capsize=3)
axA.axhline(1/3, color="k", ls=":", lw=1, alpha=.5)
axA.text(5.02, 0.34, "chance", fontsize=9, color="k", alpha=.6)
axA.set_xlabel("truth-table rows observed  (m of 9)")
axA.set_ylabel("held-out row recovery")
axA.set_xticks(ms_rows)
axA.set_title("Fourier sparsity recovers gates from few rows\n(and it is SPECIFICALLY spectral: plain ttL1 harms)", fontsize=11)
axA.legend(fontsize=8.5, loc="upper left", frameon=False)

hb = d["high_beta_m6"]
x = np.arange(len(order))
dead_hb = [hb[k]["dead"] * 100 for k in order]
axB.bar(x, dead_hb, color=[col[k] for k in order])
for i, k in enumerate(order):
    axB.text(i, dead_hb[i] + 2, f"ncE={hb[k]['ncE']:.2f}", ha="center", fontsize=8.5,
             color="#c00000" if dead_hb[i] > 50 else "#333")
axB.set_xticks(x)
axB.set_xticklabels([lab[k].split(" (")[0] for k in order], rotation=20, ha="right", fontsize=9)
axB.set_ylabel("% neurons collapsed to dead/constant")
axB.set_ylim(0, 110)
axB.set_title("Collapse-immunity at high $\\beta$\nplain ttL1 → 100% dead; corrected R$_F^\\star$ holds (floor)", fontsize=11)

fig.tight_layout()
save(fig, "rf_corrected_recovery")

print("m=6 recovery:", {k: round(d["m6"][k]["gen"], 3) for k in order})